In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master("local[*]").appName('test').getOrCreate()

26/04/12 17:03:19 WARN Utils: Your hostname, Lenovo330 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/12 17:03:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 17:03:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-04-12 11:59:34--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-12T12%3A51%3A38Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-12T11%3A50%3A54Z&ske=2026-04-12T12%3A51%3A38Z&sks=b&skv=2018-11-09&sig=gS26%2BpoIixbKZUOuIKABTsEm7ylAfQpp4zMiXo8lv%2B0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NTk5ODc3NCwibmJmIjoxNzc1OTk1MTc0LCJwYXRo

In [5]:
!wc fhvhv_tripdata_2021-01.csv  

 11908469  35725405 752335705 fhvhv_tripdata_2021-01.csv


In [6]:
df = spark.read.option("header", "true").csv('fhvhv_tripdata_2021-01.csv')

In [6]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None),
 Row(hvfhs_

In [7]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [8]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [9]:
import pandas as pd

In [10]:
df_pandas = pd.read_csv("head.csv")

In [11]:
!wc -l head.csv

1001 head.csv


In [12]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [13]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [14]:
from pyspark.sql import types

In [15]:
schema = types.StructType(
    [types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)]
    )

In [16]:
df = spark.read.option("header", "true").schema(schema).csv("fhvhv_tripdata_2021-01.csv")

In [23]:
df = df.repartition(4)

In [24]:
df.write.parquet("fhvhv/2021/01")

In [25]:
df = spark.read.parquet("fhvhv/2021/01")

In [26]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [28]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [36]:
from pyspark.sql import functions as F

In [43]:
def crazy_stuff(dispatching_base_num):
    base_num = int(dispatching_base_num[1:])
    if base_num % 7 == 0:
        return f's/{base_num:03x}'
    elif base_num % 3 == 0:
        return f'a/{base_num:03x}'
    return f'e/{base_num:03x}'

In [44]:
crazy_stuff('B1234')

'e/4d2'

In [47]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [50]:
df \
    .withColumn('pickup_date',F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_num', crazy_stuff_udf(df.dispatching_base_num)) \
    .show()

[Stage 15:>                                                         (0 + 1) / 1]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+--------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|base_num|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+--------+
|           HV0003|              B02882|2021-01-04 07:56:57|2021-01-04 08:02:23|         241|         241|   NULL| 2021-01-04|  2021-01-04|   e/b42|
|           HV0003|              B02876|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|   NULL| 2021-01-01|  2021-01-01|   e/b3c|
|           HV0003|              B02617|2021-01-02 18:17:40|2021-01-02 18:38:51|          68|         232|   NULL| 2021-01-02|  2021-01-02|   e/a39|
|           HV0003|              B02764|2021-01-03 15:32:18|2021-01-03 15:37:06|         255|         256|

In [34]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == "HV0003") \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-04 07:56:57|2021-01-04 08:02:23|         241|         241|
|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|
|2021-01-02 18:17:40|2021-01-02 18:38:51|          68|         232|
|2021-01-03 15:32:18|2021-01-03 15:37:06|         255|         256|
|2021-01-03 18:08:58|2021-01-03 18:20:32|         142|         140|
|2021-01-02 00:34:43|2021-01-02 00:45:38|          63|          77|
|2021-01-04 13:52:02|2021-01-04 13:58:38|         196|          83|
|2021-01-01 22:56:14|2021-01-01 23:06:41|          49|          80|
|2021-01-01 00:23:13|2021-01-01 00:30:35|         147|         159|
|2021-01-03 15:11:11|2021-01-03 15:30:16|         243|          24|
|2021-01-04 04:53:36|2021-01-04 05:08:13|          61|          72|
|2021-01-05 06:05:16|2021-01-05 06:16:07|       

In [32]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
